# 📊 Análise de Acessos — Secure Acess

Este notebook analisa os dados gerados pelo sistema de controle de acesso com RFID na Raspberry Pi.  
Os dados são lidos diretamente do arquivo `relatorio_acessos.csv` gerado pelo `tag.py`.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Configuração visual dos gráficos
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.facecolor'] = '#0d0d14'
plt.rcParams['figure.facecolor'] = '#0a0a0f'
plt.rcParams['axes.edgecolor'] = '#1e1e2e'
plt.rcParams['text.color'] = '#e0e0f0'
plt.rcParams['axes.labelcolor'] = '#e0e0f0'
plt.rcParams['xtick.color'] = '#5a5a7a'
plt.rcParams['ytick.color'] = '#5a5a7a'
plt.rcParams['grid.color'] = '#1e1e2e'
plt.rcParams['axes.grid'] = True

print("✓ Bibliotecas carregadas")


✓ Bibliotecas carregadas


## 📂 Carregando os dados do CSV

In [2]:
# O CSV tem duas seções: eventos detalhados e resumo final
# Vamos ler apenas os eventos (até a linha em branco)

import pandas as pd
import csv

eventos_rows = []
with open('relatorio_acessos.csv', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # Pula o cabeçalho
    for row in reader:
        # Para quando encontra linha vazia ou o resumo
        if not row or row[0] in ('', 'Resumo final', 'Nome', 'Tentativas de invasao'):
            break  # Para completamente ao chegar no resumo
        if len(row) == 5:  # Só aceita linhas com exatamente 5 colunas
            eventos_rows.append(row)

df = pd.DataFrame(eventos_rows, columns=['Data/Hora', 'Tag', 'Nome', 'Tipo', 'Status'])
df['Data/Hora'] = pd.to_datetime(df['Data/Hora'], format='%d/%m/%Y %H:%M:%S')
df['Data'] = df['Data/Hora'].dt.date

print(f"✓ {len(df)} eventos carregados")
df.head()


✓ 4 eventos carregados


,Data/Hora,Tag,Nome,Tipo,Status,Data
0,2026-05-11 20:35:56,553307625663,Julia,ENTRADA,AUTORIZADO,2026-05-11
1,2026-05-11 20:36:12,703695879170,Desconhecido,TENTATIVA_INVASAO,TAG_NAO_CADASTRADA,2026-05-11
2,2026-05-11 20:36:21,771439528262,Visitante,TENTATIVA_NEGADA,NAO_AUTORIZADO,2026-05-11
3,2026-05-11 20:36:28,357146295833,Desconhecido,TENTATIVA_INVASAO,TAG_NAO_CADASTRADA,2026-05-11


## 📌 Análise 1 — Quantas pessoas entraram por dia

In [3]:
entradas = df[df['Tipo'] == 'ENTRADA'].groupby('Data')['Nome'].count().reset_index()
entradas.columns = ['Data', 'Total de Entradas']

print("Entradas por dia:")
print(entradas.to_string(index=False))

Entradas por dia:
      Data  Total de Entradas
2026-05-11                  1


## 📌 Análise 2 — Quantas pessoas saíram por dia

In [4]:
saidas = df[df['Tipo'] == 'SAIDA'].groupby('Data')['Nome'].count().reset_index()
saidas.columns = ['Data', 'Total de Saidas']

print("Saídas por dia:")
if saidas.empty:
    print("Nenhuma saída registrada.")
else:
    print(saidas.to_string(index=False))

Saídas por dia:
Nenhuma saída registrada.


## 📌 Análise 3 — Tempo de permanência por colaborador

In [5]:
# Pareia cada ENTRADA com a SAIDA seguinte do mesmo colaborador
registros = []

for nome in df['Nome'].unique():
    df_pessoa = df[df['Nome'] == nome].sort_values('Data/Hora').reset_index(drop=True)
    entradas_p = df_pessoa[df_pessoa['Tipo'] == 'ENTRADA']['Data/Hora'].tolist()
    saidas_p   = df_pessoa[df_pessoa['Tipo'] == 'SAIDA']['Data/Hora'].tolist()

    for i in range(min(len(entradas_p), len(saidas_p))):
        duracao_min = (saidas_p[i] - entradas_p[i]).total_seconds() / 60
        registros.append({
            'Nome': nome,
            'Data': str(entradas_p[i].date()),
            'Entrada': entradas_p[i].strftime('%H:%M:%S'),
            'Saida': saidas_p[i].strftime('%H:%M:%S'),
            'Permanencia_min': round(duracao_min, 2)
        })

if registros:
    df_permanencia = pd.DataFrame(registros)
    print("Tempo de permanência por sessão:")
    print(df_permanencia.to_string(index=False))

    resumo = df_permanencia.groupby('Nome')['Permanencia_min'].sum().reset_index()
    resumo.columns = ['Nome', 'Total (min)']
    print()
    print("Total acumulado por colaborador:")
    print(resumo.to_string(index=False))
else:
    print("Nenhuma sessão completa (entrada + saída) encontrada.")

Nenhuma sessão completa (entrada + saída) encontrada.


## 📌 Análise 4 — Tentativas de acesso negado por dia

In [6]:
negados = df[df['Tipo'] == 'TENTATIVA_NEGADA'].groupby('Data')['Nome'].count().reset_index()
negados.columns = ['Data', 'Tentativas Negadas']

print("Tentativas negadas por dia:")
if negados.empty:
    print("Nenhuma tentativa negada registrada.")
else:
    print(negados.to_string(index=False))

Tentativas negadas por dia:
      Data  Tentativas Negadas
2026-05-11                   1


## 📌 Análise 5 — Tentativas de invasão por dia

In [7]:
invasoes = df[df['Tipo'] == 'TENTATIVA_INVASAO'].groupby('Data')['Tag'].count().reset_index()
invasoes.columns = ['Data', 'Tentativas de Invasão']

print("Tentativas de invasão por dia:")
if invasoes.empty:
    print("Nenhuma tentativa de invasão registrada.")
else:
    print(invasoes.to_string(index=False))

Tentativas de invasão por dia:
      Data  Tentativas de Invasão
2026-05-11                      2


## 📌 Análise 6 — Colaboradores não autorizados que mais tentaram

In [8]:
mais_tentaram = (
    df[df['Tipo'] == 'TENTATIVA_NEGADA']
    .groupby('Nome')['Data/Hora']
    .count()
    .reset_index()
    .rename(columns={'Data/Hora': 'Tentativas'})
    .sort_values('Tentativas', ascending=False)
)

print("Colaboradores não autorizados por número de tentativas:")
if mais_tentaram.empty:
    print("Nenhuma tentativa negada registrada.")
else:
    print(mais_tentaram.to_string(index=False))

Colaboradores não autorizados por número de tentativas:
     Nome  Tentativas
Visitante           1


## ✅ Resumo Geral

In [9]:
print("=" * 45)
print("        RESUMO GERAL DO SISTEMA")
print("=" * 45)
print(f"Total de eventos registrados : {len(df)}")
print(f"Entradas autorizadas         : {len(df[df['Tipo']=='ENTRADA'])}")
print(f"Saídas registradas           : {len(df[df['Tipo']=='SAIDA'])}")
print(f"Tentativas negadas           : {len(df[df['Tipo']=='TENTATIVA_NEGADA'])}")
print(f"Tentativas de invasão        : {len(df[df['Tipo']=='TENTATIVA_INVASAO'])}")
print(f"Colaboradores únicos         : {df['Nome'].nunique()}")
print("=" * 45)


        RESUMO GERAL DO SISTEMA
Total de eventos registrados : 4
Entradas autorizadas         : 1
Saídas registradas           : 0
Tentativas negadas           : 1
Tentativas de invasão        : 2
Colaboradores únicos         : 3
